Loading in libraries

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, StandardScaler, RobustScaler, KBinsDiscretizer

Setting up the panda dataframe with the dataset

In [2]:
df = pd.read_excel("data/Online_Retail.xlsx")

Task 1 (Numerical): Missing Value Imputation (sklearn.impute.SimpleImputer): For numerical streams (Quantity, UnitPrice), impute using the median strategy to prevent outlier distortion.

In [3]:
# first we split the Test and train set

X_train, X_test = train_test_split(df, test_size=0.2, random_state=21)


In [4]:
#making list of the columns we wish to transform
cols_to_transform = ["Quantity", "UnitPrice"]

#create imputer
imputer = SimpleImputer(strategy="median")

#apply transformation to the correnct columns
df[cols_to_transform] = imputer.fit_transform(df[cols_to_transform])

print(df[cols_to_transform].isnull().sum())


#### for joseph #### i need the fully transformed data set for task 2
#we make a pipeline to transform both the train and test set
# num_pipe = Pipeline(
#     steps = {
#         ("imputer", SimpleImputer(strategy="median"))
#     }
# )

# #using a column transformer to apply the transformations
# preprocessor = ColumnTransformer(
#     transformers=[
#         ("numerical", num_pipe, cols_to_transform),
#     ]
# )


# preprocessor.fit(X_train).transform(X_test)

Quantity     0
UnitPrice    0
dtype: int64


Task 2: Continuous Feature Normalization (sklearn.preprocessing): Implement StandardScaler (Z-score centering to zero mean and unit variance). Contrast performance against MinMaxScaler [0, 1] and RobustScaler (IQR), documenting sensitivity to extreme transaction anomalies

Using the preprocessed data using medain strategy for the numerical values

In [5]:
#first we import the normalization functions for the sake of not cluttering my code
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

In [6]:
#we now have to transform the data using normalization formulats
# Making copies for later comparison
df_standard = df.copy()
df_minmax = df.copy()
df_robust = df.copy()

# we then transform them using the normalization preprocessing in the order mentioned in the assignment
df_standard[cols_to_transform] = StandardScaler().fit_transform(df[cols_to_transform])
df_minmax[cols_to_transform] = MinMaxScaler().fit_transform(df[cols_to_transform])
df_robust[cols_to_transform] = RobustScaler().fit_transform(df[cols_to_transform])


Once the transformations are done we have to contrast the performance of the sensitivty to extreme anomalies

In [8]:
print("Untransformed Dataframe")
print(df[cols_to_transform].describe())

Untransformed Dataframe
            Quantity      UnitPrice
count  541909.000000  541909.000000
mean        9.552250       4.611114
std       218.081158      96.759853
min    -80995.000000  -11062.060000
25%         1.000000       1.250000
50%         3.000000       2.080000
75%        10.000000       4.130000
max     80995.000000   38970.000000


In [9]:
print("Standard Scaler transformed data")
print(df_standard[cols_to_transform].describe())

Standard Scaler transformed data
           Quantity     UnitPrice
count  5.419090e+05  5.419090e+05
mean   3.041948e-18 -1.006990e-17
std    1.000001e+00  1.000001e+00
min   -3.714426e+02 -1.143727e+02
25%   -3.921594e-02 -3.473669e-02
50%   -3.004503e-02 -2.615874e-02
75%    2.053139e-03 -4.972249e-03
max    3.713550e+02  4.027024e+02


In [10]:
print("MinMax Scaler transformed data")
print(df_minmax[cols_to_transform].describe())

MinMax Scaler transformed data
            Quantity      UnitPrice
count  541909.000000  541909.000000
mean        0.500059       0.221192
std         0.001346       0.001934
min         0.000000       0.000000
25%         0.500006       0.221124
50%         0.500019       0.221141
75%         0.500062       0.221182
max         1.000000       1.000000


In [11]:
print("Robust Scaler transformed data")
print(df_robust[cols_to_transform].describe())

Robust Scaler transformed data
            Quantity      UnitPrice
count  541909.000000  541909.000000
mean        0.728028       0.878859
std        24.231240      33.597171
min     -8999.777778   -3841.715278
25%        -0.222222      -0.288194
50%         0.000000       0.000000
75%         0.777778       0.711806
max      8999.111111   13530.527778


For the Report


As we know ZScore centers the mean to zero and standard deviation to 1. Issues with this is that Really big outliers still control the mean. As you can see in the Standard scale description for both quantity and unit price. The mean is an extremely small number for both of them means for quantity: 3.041948e-18, UnitPrice: -1.006990e-17 but the std is 1 for both of them. This means that this normalization method has high sensitivity to extreme outliers.

For the minMax normalization we run into a different problem and that is that the data stays more or less the same. since now the max is one and the min 0 extreme outliers would push most of the data to a smaller range of the space. We can see this in the MinMax scaler describe funstion. Significant portions of the data are only within the first 25% distribution line. This means that this nomalization method is has high sensitivity to outiliers.

For the Robust Scaler it tries to put most of the data between the first 3 quarters in this case 25%-75%. We can see this succesfully in quantity and not so succesfully in UnitPrice. With that being said, when compared to the original retail data the distribution has remained fairly similar. Almost scaled down by a factor of 10. Since the distribution of the data remains fairly unchanged I think that the Robust scaler has low sensitivity to outliers as the data stays similarly distributed.

Task 3: Non-Linear Feature Discretization (sklearn.preprocessing.KBinsDiscretizer): Map continuous pricing data into discrete ordinal bins. Compare (1) Uniform (equal-width), (2) Quantile (equal-frequency), and (3) K-means binning strategies.

In [13]:
from sklearn.preprocessing import KBinsDiscretizer

In [16]:
#so for this part we are using only the continuous feature which is unit price so we make a copy of that

df_copy = df.copy()
prices = df_copy[["UnitPrice"]]



In [26]:
#reading kbinDiscretizer We have to encode with ordinal to keep it as integer

uniform = KBinsDiscretizer(n_bins=12, encode="ordinal", strategy="uniform")
quantile = KBinsDiscretizer(n_bins=12, encode="ordinal", strategy="quantile")
kmeans = KBinsDiscretizer(n_bins=12, encode="ordinal", strategy="kmeans")



In [27]:
df_uniform = df.copy()
df_quantile = df.copy()
df_kmeans = df.copy()

# we then transform them using the normalization preprocessing in the order mentioned in the assignment
df_uniform[["UnitPrice"]] = uniform.fit_transform(df[["UnitPrice"]])
df_quantile[["UnitPrice"]] = quantile.fit_transform(df[["UnitPrice"]])
df_kmeans[["UnitPrice"]] = kmeans.fit_transform(df[["UnitPrice"]])


In [28]:
print("Uniform")
print(df_uniform[["UnitPrice"]].value_counts().sort_index())



Uniform
UnitPrice
0.0               2
2.0          541820
3.0              61
4.0              16
5.0               6
6.0               3
11.0              1
Name: count, dtype: int64


In [29]:
print("\nQuantile")
print(df_quantile[["UnitPrice"]].value_counts().sort_index())




Quantile
UnitPrice
0.0          45088
1.0          39541
2.0          32260
3.0          64213
4.0          13814
5.0          60570
6.0          57620
7.0          47243
8.0          37308
9.0          51543
10.0         44446
11.0         48263
Name: count, dtype: int64


In [30]:
print("\nKMeans")
print(df_kmeans[["UnitPrice"]].value_counts().sort_index())


KMeans
UnitPrice
0.0               2
1.0          540888
2.0             764
3.0             167
4.0              43
5.0              14
6.0               2
7.0               9
8.0              10
9.0               6
10.0              3
11.0              1
Name: count, dtype: int64


For Report

Comparing the 3 binning strategies yields the previous description.

The uniform method seems to have places most datapoints in bin 2 and then the rest of the bins received minimal elements. It did not even use all of the bins. It did not distribute the data well as it all stayed in one bin

The quantile method seems to have distributed the bins fairly equally. This method seems to balance how many items there are in each bin effectively balancing the entire data set.

The KMeans methods seems to have spread the population unevenly as well but every bin is being used. What this means to me is that that since kmeans explore the different means for each bin then groups items based on how much closer that mean is than the other means. Each bin is grouped with objects that have similar UnitPrice Values

Challenge B

Scenario: A data mining team constructs a 3-Nearest Neighbor (3-NN) classification model to categorize buyer behavior. Feature 1 is UnitPrice (ranging from £0.00 to £38,000.00), while Feature 2 is Quantity (ranging from -80,000 to +80,000).
Critical Prompt: If the team computes Euclidean distance \(d(x,y) = \sqrt{\sum (x_k - y_k)^2}\) on raw, unscaled features, prove geometrically why one feature
completely dominates the neighbor selection process. Explain why StandardScaler resolves this magnitude distortion and analyze how extreme outliers can compress
the variance of normal data points.

One of the features would dominate the neighbor selection process since the 2 terms do not have similar ranges in values. Lets say we have a bin that has as mean for UnitPrice as 8000 pounds and a quantity 1000 and a second bin for UnitPrice mean is 8000 and quantity mean at 50000. 

If we have an item to place in the bins with a UnitPrice of 30000 and quantity of 5000,
we would have to choose a bin by distance using Euclidean distance.  

Distance from item to bin 1 is 22360.7
Distance from item to bin 2 is 50089.9

Yet the mean price of items in bin 1 is 8,000,000 pounds and the mean price of bin 2 is 400,000,000 and the current transaction is 150,000,000 yet it's distance is closer to the cheaper bin

Standard Scaler would try to make a normal distribution out of the data meaning that both means are close to 0 and both std are 1 which means that neither of the attributes would dominate as their range has been normilized to similar distribution spaces. 

Extreme outliers compress the variance of datapoints since as shown in Task 2 they can inflate the variance of the data making many of the values fall to similar values really close to 0. 

